In [ ]:
import crunch
crunch_tools = crunch.load_notebook()
train_data, test_data=crunch_tools.load_data()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math
import os
from typing import Iterable, List, Optional, Tuple

# Import your dependencies.
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

**Visualize the data**

In [ ]:
dataset_id, x_hist, x_online, tau=train_data[2]
len_h=len(x_hist)
len_o=len(x_online)

idx_hist = np.arange(len_h)
idx_on=np.arange(len_h, len_h+len_o)

plt.figure(figsize=(15,4.5), dpi=100)
plt.grid(True, linestyle='-', alpha=0.3)
    
plt.plot(idx_hist, x_hist, color='blue', label='Historical')

if tau is not None:
    break_at=int(tau) + len_h
    x_online_pre=x_online[:tau]
    x_online_post=x_online[tau:]

    plt.axvline(x=break_at, color='red', linestyle=':', label=f'Break point at t={break_at}')
    plt.plot(range(len_h,len_h+len(x_online_pre)),x_online_pre,color='g',label='Before break')
    plt.plot(range(break_at,len_h + len_o), x_online_post, color='red', label='After break')
else:
    plt.plot(idx_on, x_online, color='green', label='No break')
plt.title(f"Series {dataset_id}| hist_len={len_h}| online_len={len_o}| tau={tau if tau is not None else "None"}")
plt.legend(loc="lower center", ncol=5, frameon=True)
plt.tight_layout()
plt.show()
plt.close('all')

Histrorical Summary Statistics

In [ ]:
def his_sum(x_hist):
  x=np.asarray(x_hist)
  n=len(x)

  mean = np.mean(x)
  median = np.median(x)
  q75, q25 = np.percentile(x, [75, 25])
  iqr = q75 - q25
  min_val = np.min(x)
  max_val = np.max(x)
  std = np.std(x, ddof=1)
  if std < 1e-8:
    std=1

  autocorr_1=pd.Series(x).autocorr(lag=1)
  autocorr_2=pd.Series(x).autocorr(lag=2)
  autocorr_3=pd.Series(x).autocorr(lag=3)
  z_scores = (x - mean) / std

  #tail probability: % of points beyond |2|, |3|, |4| std
  tail_2sigma = np.mean(np.abs(z_scores) > 2)
  tail_3sigma = np.mean(np.abs(z_scores) > 3)
  tail_4sigma = np.mean(np.abs(z_scores) > 4)

  #Skewness and Kurtosis
  skewness = float(stats.skew(x)) if n >= 3 else 0.0
  kurtosis = float(stats.kurtosis(x, fisher=True)) if n >= 4 else 0.0

  summary = {
        # Basic
        'mean': mean,
        'std': std,
        'median': median,
        'iqr': iqr,

        # Autocorrelation
        'autocorr_1': autocorr_1,
        'autocorr_2': autocorr_2,
        'autocorr_3': autocorr_3,

        # Tail probabilities
        'tail_2sigma': tail_2sigma,
        'tail_3sigma': tail_3sigma,
        'tail_4sigma': tail_4sigma,

        # Distribution shape
        'skewness': skewness,
        'kurtosis': kurtosis
    }

  return summary

Streaming Statistics

In [ ]:
class StreamingFeatureExtractor:
    def __init__(self, hist_stats, alpha=0.05):
        """
        hist_stats: dict with keys 'mean', 'std', 'tail_2sigma', 'autocorr_1'
        alpha: EWMA smoothing factor (0.02–0.20)
        """
        # Historical baselines
        self.mu_h = hist_stats['mean']
        self.sigma_h = max(hist_stats['std'], 1e-8)
        self.tail_hist = hist_stats['tail_2sigma']
        self.acf_h = hist_stats['autocorr_1']

        self.alpha = alpha

        # Streaming state
        self.ewma_mean = self.mu_h
        self.ewma_var = self.sigma_h ** 2
        self.n_eff = 0.0
        self.cusum_pos = 0.0
        self.cusum_neg = 0.0
        self.cusum_k = 0.5                     # allowance

        self.tail_window = deque(maxlen=100)   # for tail ratio
        self.prev_x = None
        self.running_acf = 0.0
        self.step = 0

    def update(self, x):
        """Process one new online point. Return feature vector of 6 numbers."""
        x = float(x)
        self.step += 1

        # 1. EWMA mean
        self.ewma_mean = (1 - self.alpha) * self.ewma_mean + self.alpha * x

        # 2. EWMA variance (using residual from running mean)
        residual = x - self.ewma_mean
        self.ewma_var = (1 - self.alpha) * self.ewma_var + self.alpha * (residual * residual)

        # 3. Effective sample size
        self.n_eff = (1 - self.alpha) * self.n_eff + 1.0

        # 4. CUSUM (standardized by historical mean/std)
        z = (x - self.mu_h) / self.sigma_h
        self.cusum_pos = max(0.0, self.cusum_pos + z - self.cusum_k)
        self.cusum_neg = max(0.0, self.cusum_neg - z - self.cusum_k)

        # 5. Tail ratio (rolling window of raw values)
        self.tail_window.append(x)
        if len(self.tail_window) >= 20:
            n_extreme = sum(1 for v in self.tail_window if abs(v - self.mu_h) > 2 * self.sigma_h)
            tail_current = n_extreme / len(self.tail_window)
        else:
            tail_current = 0.0

        # 6. Running autocorrelation (lag 1) via EWMA of product
        if self.prev_x is not None:
            prod = x * self.prev_x
            self.running_acf = (1 - self.alpha) * self.running_acf + self.alpha * prod
        self.prev_x = x

        # ----- Feature calculations (clipped to reasonable ranges) -----
        # Mean z‑score
        se_mean = self.sigma_h / math.sqrt(max(self.n_eff, 1.0))
        z_mean = (self.ewma_mean - self.mu_h) / max(se_mean, 1e-8)

        # Variance ratio (centered at 1, so 0 = no change)
        var_ratio = self.ewma_var / (self.sigma_h * self.sigma_h + 1e-8)

        # Normalized CUSUM
        norm = self.sigma_h * math.sqrt(max(self.n_eff, 1.0))
        cusum_pos_norm = self.cusum_pos / max(norm, 1e-8)
        cusum_neg_norm = self.cusum_neg / max(norm, 1e-8)

        # Tail difference (scaled by historical tail probability)
        tail_diff = (tail_current - self.tail_hist) / max(self.tail_hist + 0.01, 0.01)

        # Autocorrelation difference (absolute deviation)
        acf_diff = abs(self.running_acf - self.acf_h)

        # Clip and return
        return [
            max(-10.0, min(10.0, z_mean)),                    # 0: mean z-score
            max(-5.0, min(10.0, var_ratio - 1.0)),           # 1: variance shift (centered)
            max(0.0, min(20.0, cusum_pos_norm)),             # 2: positive CUSUM evidence
            max(0.0, min(20.0, cusum_neg_norm)),             # 3: negative CUSUM evidence
            max(-3.0, min(10.0, tail_diff)),                 # 4: tail ratio change
            max(-5.0, min(5.0, acf_diff))                    # 5: autocorrelation change
        ]

Rolling PCA detector

In [ ]:
class RollingPCADetector:
    def __init__(self, window_size=50, update_every=5, n_components=1):
        self.window_size = window_size
        self.update_every = update_every
        self.n_components = n_components
        self.feature_window = deque(maxlen=window_size)
        self.step = 0
        self.last_prob = 0.0
        self.baseline_pca = None

    def set_baseline(self, historical_feature_vectors):
        """Fit PCA on historical (no‑break) feature vectors."""
        if len(historical_feature_vectors) < self.window_size:
            # Not enough data, use a dummy baseline
            self.baseline_pca = PCA(n_components=self.n_components)
            self.baseline_pca.fit(np.random.randn(self.window_size, len(historical_feature_vectors[0])))
        else:
            self.baseline_pca = PCA(n_components=self.n_components)
            self.baseline_pca.fit(historical_feature_vectors)

    def update(self, feature_vector):
        """Add new feature vector, return break probability (0..1)."""
        self.feature_window.append(feature_vector)
        self.step += 1
        if len(self.feature_window) < self.window_size or self.baseline_pca is None:
            return 0.0
        if self.step % self.update_every != 0:
            return self.last_prob
        # Fit PCA on current window
        current_pca = PCA(n_components=self.n_components)
        current_pca.fit(np.array(self.feature_window))
        # Compute angle between first principal components
        v1 = self.baseline_pca.components_[0]
        v2 = current_pca.components_[0]
        dot = np.clip(np.abs(np.dot(v1, v2)), -1.0, 1.0)
        angle = np.arccos(dot)               # 0 … π/2
        prob = angle / (np.pi / 2)           # 0 … 1
        self.last_prob = prob
        return prob

In [ ]:
#Train function
def train(datasets, model_directory_path):
    """
    datasets: list of (dataset_id, x_hist, x_online, tau)
    For each series, collect feature vectors from the historical segment
    and also from the pre‑break part of the online segment (if tau given).
    Then fit baseline PCA on all those vectors.
    """
    all_historical_vectors = []
    for _, x_hist, x_online, tau in datasets:
        # compute historical stats
        hist_stats = his_sum(x_hist)
        # create streaming extractor
        extractor = StreamingFeatureExtractor(hist_stats, alpha=0.05)
        # extract features from historical segment (no break)
        for x in x_hist:
            vec = extractor.update(x)
            all_historical_vectors.append(vec)
        # also from online segment BEFORE the break (if break exists)
        if tau is not None and tau > 0:
            # reset extractor to initial state? Actually we want features from online pre‑break
            # Better to create a fresh extractor for the online segment using same hist_stats
            online_extractor = StreamingFeatureExtractor(hist_stats, alpha=0.05)
            for i, x in enumerate(x_online):
                if i < tau:
                    vec = online_extractor.update(x)
                    all_historical_vectors.append(vec)
                else:
                    break
    # Fit baseline PCA on all collected vectors
    if len(all_historical_vectors) < 50:
        # fallback: generate dummy vectors
        all_historical_vectors = np.random.randn(50, 6).tolist()
    pca_detector = RollingPCADetector(window_size=50, update_every=5, n_components=1)
    pca_detector.set_baseline(all_historical_vectors)
    # Save model
    os.makedirs(model_directory_path, exist_ok=True)
    joblib.dump(pca_detector, os.path.join(model_directory_path, "pca_detector.joblib"))
    # Also save the window_size and update_every for inference
    params = {'window_size': 50, 'update_every': 5, 'n_components': 1}
    joblib.dump(params, os.path.join(model_directory_path, "params.joblib"))

In [ ]:
#Infer function
def infer(datasets, model_directory_path):
    # Load trained PCA detector and parameters
    pca_detector = joblib.load(os.path.join(model_directory_path, "pca_detector.joblib"))
    params = joblib.load(os.path.join(model_directory_path, "params.joblib"))
    window_size = params['window_size']
    update_every = params['update_every']
    n_components = params['n_components']

    yield   # ready signal

    for x_historical, x_online in datasets:
        # Step 1: historical summary
        hist_stats = his_sum(x_historical)
        # Step 2: streaming feature extractor
        extractor = StreamingFeatureExtractor(hist_stats, alpha=0.05)
        # Step 3: rolling PCA (new instance per series, but reuse baseline from training)
        rolling_pca = RollingPCADetector(window_size, update_every, n_components)
        rolling_pca.baseline_pca = pca_detector.baseline_pca   # share baseline
        # Process each online point
        for x in x_online:
            feat_vec = extractor.update(x)
            prob = rolling_pca.update(feat_vec)
            yield float(prob)

**Local Test**

In [ ]:
crunch.test(
    # Uncomment to disable the train
    #force_first_train=False,

    # Uncomment to disable the determinism check
    # no_determinism_check=True,
)